# Linear regression
In this notebook,  we demonstrate how linear regression can be used to estimate the parameters of the Gutenberg–Richter law from observed earthquake catalog data, and discuss the temporal variability of the fitted parameters.

## Science Question: Estimating earthquake frequency using Gutenberg-Richter law <a id="G_R_law"></a>

The Gutenberg-Richter (G-R) law describes a linear relationship between the logarithm of the number of earthquakes (earthquake frequency) and the magnitude of earthquakes. The G-R law can be expressed mathematically as:

$\log_{10} N = a - bM$,

where $a$ and $b$ are constants and depend on the region and the time period considered, $N$ is the number of earthquakes with magnitude greater than or equal to $M$.

The b-value, $b$ in the above equation, indicates the ratio of small to large earthquakes. A lower b-value indicates a higher likelihood of larger earthquakes and vice-versa. This value is often used to understand the seismicity of a region and, as such, in probabilistic hazard prediction models.

The a-value, $a$, is a proxy for the background rate of seismicity rate of the region.

<div align="center">
    <img src='./images/G-R_subduction_zones.png' width="400"/>
    <figcaption> Frequency-magnitude distribution for earthquakes deeper than 500 km in the South America (SA), PH (Philippine), JK (Japan-Kuril), and IBM (Izu-Bonin-Mariana) subduction zones (dots) along with their best-fitting lines based on the G-R law. Earthquake magnitudes within the Tonga slab were partitioned at Mw = 6. Image: Zhan, 2017 </figcaption>
</div>

The deviation in earthquake frequency for the lower-magnitude earthquakes in the above figure can be due to several reasons, including an incomplete earthquake catalog for the time period considered and/or the threshold detection limits of the seismic instrument. If the catalog is deemed incomplete for magnitudes lower than the detection threshold, then this data is dropped before regression. This threshold is typically referred to as the **magnitude of completeness (Mc)**, which is the lowest magnitude above which the catalog is assumed to record all events reliably.

But overall, we can estimate a log-linear relationship between earthquake frequency and magnitude. In the next section, we will show how to compute the best-fitting line for a given earthquake catalog and estimate the b-value for a specific region.

## Machine Learning Technique

If the relationship between two independent variables in a given data is modeled as a linear function, the line that best describes this relationship can be computed using linear regression. The best-fitting line is the one that minimizes the sum of squared differences between the observed values and the predicted values from the linear model. This can be visualized using the following figure:

<div align="center">
    <img src='./images/linear_regression.png' width="400"/>
    <figcaption> Linear regressions minimizes the residuals between the best fit line (red) and the data, y_n (gray). MSE is the mean squared error of the differences, delta y_n (dashed red line). MSE: mean squared error. Image modified from Pyrcz (2024) </figcaption>
</div>

### Analytical solution

For a given set of points, $x_i, y_i$, the mean squared residual (MSE) can be expressed as:

$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - (mx_i + c))^2$$

where $m$ is the slope of the line, $c$ is the y-intercept, and $n$ is the number of data points.

To find the best-fitting line, we can take the derivative of the MSE with respect to $m$ and $c$, set them to zero, and solve for $m$ and $c$. This gives:

\begin{equation}
    \frac{\partial \text{MSE}}{\partial m} = \frac{1}{n} \frac{\partial}{\partial m} \sum_{i=1}^{n} (y_i - (mx_i + c))^2 = \frac{1}{n} \sum_{i=1}^{n} - 2 x_i (y_i - c - mx_i), \tag{1}
\end{equation}

and
\begin{equation}
    \frac{\partial {}}{\partial c} = \frac{1}{n} \frac{\partial}{\partial c}  \sum_{i=1}^{n} (y_i - (mx_i + c))^2 = \frac{1}{n} \sum_{i=1}^{n} - 2 (y_i - c - mx_i)
    \tag{2}
\end{equation}

Setting equation (1) and (2) to zero:

\begin{equation}
  \sum_{i=1}^{n} x_i (y_i - c - mx_i) = 0 \qquad \text{and} \qquad \sum_{i=1}^{n} (y_i - c - mx_i) = 0 \tag{3}
\end{equation}

Expanding them:

\begin{equation}
  \sum_{i=1}^{n} x_i y_i - c\sum_{i=1}^{n} x_i - m\sum_{i=1}^{n} x_i^2 = 0  \tag{4}
\end{equation}

\begin{equation}
  \sum_{i=1}^{n} y_i - nc - m\sum_{i=1}^{n} x_i = 0 \tag{5}
\end{equation}

In equation (5), we solve for $c$:

\begin{equation}
  c = \bar{y} - m\bar{x} \tag{6}
\end{equation}

where $\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$ and $\bar{y} = \frac{1}{n}\sum_{i=1}^{n} y_i$ are the sample means of $x$ and $y$.

Substituting this expression for $c$ into equation (4) and solving for $m$:

\begin{equation}
  m = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n} (x_i - \bar{x})^2}, \tag{7}
\end{equation}


Equation (7) suggests that best-fit line minimizing the MSE has slope $m$ equal to the covariance of $x$ and $y$ divided by the variance of $x$, and intercept $c$ chosen such that the line passes through data mean $(\bar{x}, \bar{y})$ (Equation 6).

## Python Implementation

### Step 1: Explore the data

The accompanying earthquake catalog, downloaded from the [USGS catalog](https://earthquake.usgs.gov/earthquakes/search/), is available in the `data` folder. The catalog contains earthquake events from 2000 to 2026 with magnitude > 4.5 in Japan and Chile. We investigate the catalog using pandas dataframe.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datetime import datetime

In [ ]:
# inspect the data
df_eqs = pd.read_csv('data/chile_earthquakes.csv')
df_eqs.info()
df_eqs.head()

We are interested in the magnitude and the number of earthquakes with magnitude greater than or equal to a value for a time period.
Therefore, we only retain columns with features **time** and **mag**.

In [ ]:
cols_to_retain =['time','mag']
df_eqs         = df_eqs[cols_to_retain]

# We extract only the year from the time column, which we use to later
# group the earthquake events.
# This is done so that we have enough events in each group for a meaningful
# b-value interpretation.
# We also sort the data by time to analyze the temporal variation within the data.
df_eqs['time'] = pd.to_datetime(df_eqs['time']).dt.year
sort_dataset = lambda x : x.sort_values(by='time').reset_index(drop=True)
df_sorted = sort_dataset(df_eqs)

In [ ]:
df_sorted.head()

#### Variation over time
Let's explore how the number of earthquakes recorded has changed over time. The G-R law assumes that the earthquakes recorded over a given time period does not change, but in reality, the number of earthquakes recorded can vary due to changes in the seismic network, detection capabilities, and other factors e.g., post vs. interseismic periods. This is evidenced immediately after an earthquake, when frequency of a sequence of after-shocks decays with time, i.e., following Omori's law (e.g., Shaw 1993).


In [ ]:
fig, ax = plt.subplots(1,1, figsize=(4,4))

df_sorted.plot(ax=ax, kind='scatter', x='time', y='mag', legend=False, s=2)
ax.set_xlabel('Year')
ax.set_ylabel('Magnitude')
plt.tight_layout()

The above figure qualitatively shows that the earthquake distribution has roughly been constant over this time period. Hence, we investigate their magnitude distribution for M>2, during this time to estimate the b-value.

#### Variation over magnitude
We can group the earthquakes by magnitude and count the number of events in each group (bin). Within each bin, we count all events at or above a given magnitude (i.e., binned cumulatively). Such representation will allow us to visualize the frequency-magnitude distribution and assess whether it follows a linear trend as predicted by the Gutenberg-Richter law.

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(4,4))

# Get cumulative counts (N >= M), where M is the earthquake magnitude spaced at 0.1 intervals
mag_bins  = np.arange(df_sorted['mag'].min(), df_sorted['mag'].max() + 0.2, 0.2)
counts, _ = np.histogram(df_sorted['mag'], bins=mag_bins)
N_counts  = np.cumsum(counts[::-1])[::-1]

# Ignore the last bin as it is only used
ax.plot(mag_bins[:-1], N_counts, '-o')
ax.set_yscale('log')
ax.set_xlabel('Magnitude')
ax.set_ylabel('Earthquake Frequency (N ≥ M)')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

The plot above shows:

- a plateau for lower magnitudes, likely  due to incomplete earthquake catalog.
- a kink at higher magnitudes, that could be due to the chosen bins, limited number of large-magnitude earthquakes in the time period considered, or representative of a physical anomaly to sustain high-magnitude earthquakes in the region.

### Step 2 : Regression construction


In [ ]:
def fit_gr_regressor(x, y):
    """
    Following Eq. 7 for slope and Eq. 6 for intercept in the above
    explanation.
    """
    x_bar = np.mean(x)
    y_bar = np.mean(y)

    m = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar) ** 2)
    c = y_bar - m * x_bar

    return m, c

In [ ]:
X = mag_bins[:-1]
y = np.log10(N_counts)

b_value, a_value = fit_gr_regressor(X, y)

print(f"b-value: {b_value:.6f}")
print(f"a-value: {a_value:.6f}")

> <b>Note:</b> The above linear fit can also be done simply using the numpy function <code>np.polyfit</code> [function](https://numpy.org/doc/2.3/reference/generated/numpy.polyfit.html), giving the same result as the analytical solution above.


### Step 3: Plot data and model

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4))

# observed data
ax.plot(mag_bins[:-1], N_counts, 'o', color='k', ms=4, label='Observed')
# fitted regression line
N_polyfit = 10 ** (b_value * X + a_value)

ax.plot(X, N_polyfit, '-', color='crimson', lw=2, label=f'Regression: b={b_value:.2f}')

ax.set_yscale('log')
ax.set_xlabel('Magnitude')
ax.set_ylabel('Earthquake Frequency (N ≥ M)')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

The fitted b-value, 0.79, deviates fom the typical b-value of 1 for most tectonic regions. This is because the catalog is not complete for lower magnitudes, and the limited number of large-magnitude earthquakes in the time period considered. We remove these outliers and fit the G-R law again in the next exercise.

### Exercise: Find the magnitude of completeness
Notice how the best-fitting line deviates from the observed data for lower (M < 4) and large (~ M > 8.2) magnitudes. The G-R relation only holds linearly above Mc (magnitude of completeness, see the [above section](G_R_law)), where the catalog stops missing small events.

- To find the lower magnitude cutoff, we can use the maximum value of the histogram. This makes sense because based on the G-R law, we expect the number of earthquakes to increase with decreasing magnitude and even though data may be missing for lower magnitudes, the number of earthquakes should still be increasing. Therefore, the maximum value of the histogram is a good estimate for Mc. You can get the magnitude corresponding to the maximum counts using the following code:

```python
idx_max = np.argmax(counts)
Mc      = mag_bins[idx_max]
```

- We also drop the highest magnitude bins where N becomes too small (i.e., N < 2) to avoid the noisy tail dominating the fit. The total mask would then be

```python
mask = (mag_bins >= Mc) & (counts >= 2)
```

- Now, you can fit the G-R law on the data subset,
    
```python
x_fit = mag_bins[:-1][fit_mask]
y_fit = np.log10(N_counts[fit_mask])
```

### Exercise: Investigate temporal variation of b-value
After we have computed the b-value using the earthquake catalog, we can go a step further in evaluating whether the b-value changes over time or not. This is important as several studies (e.g., Tsukakoshi and Shimazaki, 2003) have shown that b-value tends to decrease as differential stress increases, and higher stress favors the more frequent occurrence of larger-magnitude events relative to smaller ones. Therefore, a drop in b-value preceding a large earthquake can be interpreted as an indicator for stress accumulation and possible fault criticality, while a jump in b-value afterwards reflects the stress-relaxation that follows a major rupture.

To study the temporal variability, we first split the catalog data into **train**(ing) and **test**(ing). Then, we get the b-value from regression on the training data. Finally, we check how well it forecasts seismicity rates in a subsequent test window. If the fit is very different for the test data, it suggests the b-value shifted between the two periods, potentially due to a stress-state change from a large event, a swarm, or some other process affecting the tectonic stresses.


In [ ]:
def fit_window(df, bin_width=0.2, min_count=2):
    """
    Fit the Gutenberg-Richter law to a given dataframe of earthquake magnitudes.
    Returns the slope (b-value), intercept (a-value), earthquake magnitude bins,
    and cumulative counts.
    """

    mags = df['mag'].values
    mag_bins  = np.round(np.arange(mags.min(), mags.max() + bin_width, bin_width), 2)
    counts, _ = np.histogram(mags, bins=mag_bins)
    N_counts  = np.cumsum(counts[::-1])[::-1]

    m = mag_bins[:-1]
    Mc      = mag_bins[np.where(counts==np.max(counts))]

    # ignore very low- and large-magnitude earthquakes using
    # a mask. We covered this in the exercise above.
    valid = (m >= Mc) & (N_counts >= min_count)

    slope, intercept = fit_gr_regressor(m[valid], np.log10(N_counts[valid]))

    return dict(b=slope, a=intercept, mag_bins=mag_bins, N_counts=N_counts)

In [ ]:
# Define the time period to split the dataset into training and testing sets.
split_year = 2023

train_df = df_sorted[df_sorted['time'] <= split_year]
test_df  = df_sorted[df_sorted['time'] > split_year]

print(f"Split year: {split_year}")
print(f"Train: {len(train_df)} events ({train_df['time'].min()}-{train_df['time'].max()})")
print(f"Test:  {len(test_df)} events ({test_df['time'].min()}-{test_df['time'].max()})")

train_fit = fit_window(train_df)
test_fit  = fit_window(test_df)

print(f"\nTrain b-value: {train_fit['b']:.3f}, a-value: {train_fit['a']:.3f}")
print(f"Test  b-value: {test_fit['b']:.3f}, a-value: {test_fit['a']:.3f}")

The higher b-value in the test data suggests that in the time period considered for the test data (2024–2025), there are fewer large-magnitude earthquakes relative to the small ones than the training data time period (2000–2023) would predict. This could be because our test window is only ~ 2 years, and given that there are less large-magnitude earthquake in a given time period (G-R law), our test data shows the sampling bias and not an actual stress-regime change.

#### Predict G-R line for test dataset
The a-value depends on the total number of earthquakes in a given time period. Assuming that this number is proportional to the length of the time period, we can predict the number of earthquakes in a test period based on the a-value computed from a training period.

Expressing G-R law as $ N = N_o 10^{-bM} $, where $N_o = 10^a$ is the total number of events, we can calculate the a-value for the test dataset as, $ a_{test} =\log_{10}(time_{test}/time_{train}) + a_{train} $

In [ ]:
train_years = train_df['time'].max() - train_df['time'].min()
test_years  = test_df['time'].max() - test_df['time'].min()
rate_scale  = test_years / train_years

In [ ]:
# G-R law prediction using the train fit parameters, for the test data set.
a_predicted = train_fit['a'] + np.log10(rate_scale)
m_range     = test_fit['mag_bins'][:-1]
N_predicted = 10 ** (a_predicted + train_fit['b'] * m_range)

# Plot the observed and predicted earthquake frequency-magnitude distributions for
# the train and test datasets.
fig, ax = plt.subplots(1, 1, figsize=(4, 4))

ax.plot(train_fit['mag_bins'][:-1], train_fit['N_counts'], 'o',
        ms=4, alpha=0.5, label=f"Train observed ({train_df['time'].min()}-{train_df['time'].max()})")
ax.plot(test_fit['mag_bins'][:-1], test_fit['N_counts'], 'o',
        ms=4, alpha=0.5, label=f"Test observed ({test_df['time'].min()}-{test_df['time'].max()})")
ax.plot(m_range, N_predicted, '--', lw=2, label='Predicted from training set')

ax.set_yscale('log')
ax.set_xlabel('Magnitude')
ax.set_ylabel('Cumulative Frequency (N ≥ M)')
ax.grid(True, which='both', alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

The b-value derived from the training data fits most of the test data across the mid-magnitude range (M = 4.5-6.5). However, deviations exist at low-mid magnitudes (M < 4.5) and high magnitudes (M > 6.5),  that could be due to incomplete earthquake catalog as we only fit the data over two years time period.

### Try it out yourself

- Compute the b-value for different time periods and see if there are any significant changes in seismicity over time. What does this tell you about the temporal variation of seismicity in the region?

- The accompanying data folder has earthquake catalogs for Japan as well. Can you compute the b-value for Japan and compare it with the b-value for Chile? What does this tell you about the seismicity of the two regions?

- You can explore b-value of various regions simply by downloading the earthquake catalog from the [USGS website](https://earthquake.usgs.gov/earthquakes/search/)

#### References:

- Shaw, B. E. (1993). Generalized Omori law for aftershocks and foreshocks from a simple dynamics. Geophysical research letters, [20(10), 907-910](https://agupubs.onlinelibrary.wiley.com/doi/abs/10.1029/93GL01058).
- Tsukakoshi, Y., & Shimazaki, K. (2008). Decreased b-value prior to the M 6.2 Northern Miyagi, Japan, earthquake of 26 July 2003. Earth, Planets and Space, [60(9), 915-924](https://link.springer.com/article/10.1186/BF03352847).
- Zhan, Z. (2017). Gutenberg–Richter law for deep earthquakes revisited: A dual-mechanism hypothesis. Earth and Planetary Science Letters, [461, 1-7](https://www.sciencedirect.com/science/article/pii/S0012821X16307439).
- O'Connor, P. (2024). Earthquake Modeling - Rates and Magnitudes. [Kaggle](https://www.kaggle.com/code/wumanandpat/earthquake-modeling-rates-and-magnitudes)
- Pyrcz, M.J., 2024, Applied Machine Learning in Python: A Hands-on Guide with Code [e-book](https://geostatsguy.github.io/MachineLearningDemos_Book/intro.html). Zenodo. doi:10.5281/zenodo.15169139

 &nbsp;<div style="text-align: right">  
    &rarr; <b>NEXT: [Polynomial regression on well log data](./2_polynomial_regression.ipynb) </b> <a href=""></a> &nbsp;&nbsp;
     <img src="../../../assets/education-gem-notebooks_icon.png" alt="icon"  style="width:4%">
  </div>